<a href="https://colab.research.google.com/github/redditcommunitydiscussions-collab/Omaha-Forensic-Engine/blob/main/Stock_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 Alpha Engine - Module 1: The Macro Filter (v2)
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Select your desired `Run Mode`. The filters will adjust automatically.
#@markdown 2.  Enter your Google Sheet ID. The results will be saved to a tab named `Filtered_Universe`.

#@markdown ---
#@markdown ### **Select Run Mode**
RUN_MODE = "Sleeping Giant (Finds established large-caps with hidden growth)" #@param ["10x Compounder (Finds high-growth small/mid-caps)", "Sleeping Giant (Finds established large-caps with hidden growth)"]

#@markdown ---
#@markdown ### **Google Sheet Configuration**
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
OUTPUT_TAB_NAME = "Filtered_Universe" #@param {type:"string"}

#@markdown ---
#@markdown ### **Filters for "10x Compounder" Mode**
#@markdown These filters are only active when "10x Compounder" mode is selected.
MARKET_CAP_MIN_B = 0.1 #@param {type:"number"}
MARKET_CAP_MAX_B = 100.0 #@param {type:"number"}
MIN_AVG_DOLLAR_VOLUME_M = 1.0 #@param {type:"number"}

#@markdown ---
#@markdown ### 2. Click the "Play" button to run the filter.

# ==============================================================================
# The code below this line executes the analysis logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install and Import Libraries
!pip -q install yfinance gspread google-auth gspread_dataframe pandas numpy tqdm

import pandas as pd
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
import time
import requests
import io
import yfinance as yf
import re

# Step 2: Authenticate with Google
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Step 3: Helper Functions
def fetch_all_us_tickers():
    """Fetches a list of all US-listed common stocks, applying filters to remove non-common shares."""
    urls = {
        'nasdaq': 'https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt',
        'other': 'https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt'
    }
    all_tickers = []
    for name, url in urls.items():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        content = "\n".join([line for line in response.text.splitlines() if "File Creation Time:" not in line])
        df = pd.read_csv(io.StringIO(content), sep='|')

        # Robust filtering for common stocks
        if 'ETF' in df.columns:
            df = df[df['ETF'] == 'N']
        if 'Test Issue' in df.columns:
            df = df[df['Test Issue'] == 'N']
        if name == 'nasdaq' and 'Financial Status' in df.columns:
             # This file has a useful status flag. 'N' is for normal, active common stocks.
            df = df[df['Financial Status'] == 'N']

        symbol_col = 'Symbol' if 'Symbol' in df.columns else 'ACT Symbol'

        # Additional filter to remove symbols that are likely not common stocks (e.g., warrants, preferred)
        symbols = df[symbol_col].dropna().tolist()
        clean_symbols = [s for s in symbols if not re.search(r'[\.\$\^]', str(s))]
        all_tickers.extend(clean_symbols)

    return sorted(list(set(all_tickers)))

def get_stock_data_batch(tickers):
    """Fetches key data for a batch of tickers and returns a DataFrame."""
    data = []
    chunk_size = 100
    for i in tqdm(range(0, len(tickers), chunk_size), desc="Fetching Market Data"):
        chunk = tickers[i:i+chunk_size]

        # Download historical data for the chunk, grouping by ticker for resilience
        # Added auto_adjust=True to address the FutureWarning
        hist_data = yf.download(chunk, period="1y", progress=False, threads=True, group_by='ticker', auto_adjust=True)

        if hist_data.empty:
            print(f"Warning: Could not download historical data for chunk starting with {chunk[0]}")
            continue

        info_data = yf.Tickers(chunk)

        for ticker in chunk:
            try:
                # Check if we have valid historical data for this ticker
                if ticker not in hist_data.columns.get_level_values(0) or hist_data[ticker].empty:
                    continue

                info = info_data.tickers[ticker].info
                # Use 'Close' since auto_adjust=True handles adjustments
                hist_close = hist_data[ticker]['Close']
                hist_volume = hist_data[ticker]['Volume']

                if hist_close.isnull().all() or len(hist_close.dropna()) < 250:
                    continue

                market_cap = info.get('marketCap')
                dollar_volume = hist_close * hist_volume
                avg_dollar_volume = dollar_volume.rolling(window=50).mean().iloc[-1]
                sector = info.get('sector')
                industry = info.get('industry')

                if market_cap and avg_dollar_volume and sector:
                    data.append({
                        'Ticker': ticker,
                        'Market Cap': market_cap,
                        'Avg Dollar Volume': avg_dollar_volume,
                        'Sector': sector,
                        'Industry': industry
                    })
            except Exception:
                continue
    return pd.DataFrame(data)


def run_filter_and_save(sh, output_tab_name, run_mode):
    """Main function to orchestrate the filtering and saving process."""
    if run_mode.startswith("10x Compounder"):
        print("--- Running in '10x Compounder' Mode ---")
        market_cap_min = MARKET_CAP_MIN_B * 1e9
        market_cap_max = MARKET_CAP_MAX_B * 1e9
        min_avg_dollar_volume = MIN_AVG_DOLLAR_VOLUME_M * 1e6
        industry_keywords = [
            'software', 'semiconductor', 'health', 'biotechnology', 'internet',
            'technology', 'diagnostics', 'robotics', 'ai'
        ]
    else: # Sleeping Giant Mode
        print("--- Running in 'Sleeping Giant' Mode ---")
        market_cap_min = 50 * 1e9
        market_cap_max = 500 * 1e9
        min_avg_dollar_volume = 100 * 1e6 # Higher liquidity bar
        industry_keywords = [
            'software—infrastructure', 'semiconductors', 'it services', 'communication equipment'
        ]

    print("Step 1: Fetching all US stock tickers...")
    all_tickers = fetch_all_us_tickers()
    print(f"Found {len(all_tickers)} potential common stock tickers.")

    print("\nStep 2: Fetching market data for all tickers (this can take a while)...")
    market_data_df = get_stock_data_batch(all_tickers)
    print(f"Successfully retrieved data for {len(market_data_df)} tickers.")

    # --- ROBUSTNESS CHECK ---
    if market_data_df.empty:
        print("\n❌ Error: Failed to retrieve any valid market data. The yfinance API might be temporarily unavailable or blocking requests. Please try again later.")
        return # Stop execution gracefully

    print("\nStep 3: Applying filters...")
    # Filter by Market Cap
    filtered_df = market_data_df[
        (market_data_df['Market Cap'] >= market_cap_min) &
        (market_data_df['Market Cap'] <= market_cap_max if market_cap_max else True)
    ]
    print(f"  {len(filtered_df)} stocks passed the market cap filter.")

    # Filter by Liquidity
    filtered_df = filtered_df[filtered_df['Avg Dollar Volume'] >= min_avg_dollar_volume]
    print(f"  {len(filtered_df)} stocks passed the liquidity filter.")

    # Filter by Industry
    filtered_df = filtered_df[
        filtered_df['Industry'].str.lower().str.contains('|'.join(industry_keywords), na=False)
    ]
    print(f"  {len(filtered_df)} stocks passed the industry filter.")

    final_count = len(filtered_df)
    if final_count > 0:
        print(f"\n✅ Filtering complete. Found {final_count} high-potential candidates.")
        try:
            print(f"Writing results to the '{output_tab_name}' tab...")
            ws_out = None
            try:
                ws_out = sh.worksheet(output_tab_name)
                ws_out.clear()
            except gspread.exceptions.WorksheetNotFound:
                ws_out = sh.add_worksheet(output_tab_name, rows=final_count + 5, cols=10)

            set_with_dataframe(ws_out, filtered_df, include_index=False, include_column_header=True)
            print("✅ Successfully wrote data to Google Sheets.")
        except Exception as e:
            print(f"❌ An error occurred while writing to Google Sheets: {e}")
    else:
        print("\n⚠️ No stocks passed all the filters. The output sheet was not updated.")

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 1 ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Successfully connected to Google Sheet: '{sh.title}'")
    run_filter_and_save(sh, OUTPUT_TAB_NAME, RUN_MODE)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Spreadsheet not found. Please check your SHEET_ID.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 1 Complete ---")



--- Starting Alpha Engine: Module 1 ---
✅ Successfully connected to Google Sheet: 'Stock Tracker'
--- Running in 'Sleeping Giant' Mode ---
Step 1: Fetching all US stock tickers...
Found 6440 potential common stock tickers.

Step 2: Fetching market data for all tickers (this can take a while)...


Fetching Market Data:   0%|          | 0/65 [00:00<?, ?it/s]

Successfully retrieved data for 1350 tickers.

Step 3: Applying filters...
  91 stocks passed the market cap filter.
  86 stocks passed the liquidity filter.
  6 stocks passed the industry filter.

✅ Filtering complete. Found 6 high-potential candidates.
Writing results to the 'Filtered_Universe' tab...
✅ Successfully wrote data to Google Sheets.

--- Module 1 Complete ---


In [ ]:
#@title Quick Save Cell - This cell will save the results from your last run without re-filtering.

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default

# --- ACTION REQUIRED: Paste your correct Sheet ID below ---
# The ID is the long string in the URL of your sheet between "/d/" and "/edit".
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU"
OUTPUT_TAB_NAME = "Filtered_Universe"
# ---------------------------------------------------------

print("Attempting to connect to your Google Sheet...")

try:
    # Re-authenticate and connect using the correct ID
    auth.authenticate_user()
    SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
    creds, _ = default(scopes=SCOPES)
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SHEET_ID)

    print(f"✅ Successfully connected to Google Sheet: '{sh.title}'")

    # Save the 'filtered_df' that already exists in memory
    print(f"Saving the {len(filtered_df)} stocks found in the last run...")

    try:
        ws_out = sh.worksheet(OUTPUT_TAB_NAME)
        ws_out.clear()
    except gspread.exceptions.WorksheetNotFound:
        ws_out = sh.add_worksheet(OUTPUT_TAB_NAME, rows=1000, cols=20)

    set_with_dataframe(ws_out, filtered_df, include_index=False, include_column_header=True)
    print(f"✅ Success! Your results are now in the '{OUTPUT_TAB_NAME}' tab.")

except Exception as e:
    print(f"❌ An error occurred. Please double-check your Sheet ID and sharing permissions.")
    print(f"   Error details: {e}")

Attempting to connect to your Google Sheet...
✅ Successfully connected to Google Sheet: 'Stock Tracker'
❌ An error occurred. Please double-check your Sheet ID and sharing permissions.
   Error details: name 'filtered_df' is not defined


In [ ]:
#@title 🚀 Alpha Engine - Module 2: The Triple Scorer (SIQ + CAN SLIM + GARP)
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  This module will read tickers from the `Filtered_Universe` tab.
#@markdown 3.  The results will be saved to a new tab: `Triple_Scores`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
INPUT_TAB_NAME = "Filtered_Universe" #@param {type:"string"}
OUTPUT_TAB_NAME = "Triple_Scores" #@param {type:"string"}

#@markdown ---
#@markdown ### 2. Click the "Play" button to run the scorer.
#@markdown This will calculate the Quality (SIQ), Momentum (CAN SLIM), and Value (GARP) scores for each stock.

# ==============================================================================
# The code below this line executes the analysis logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install yfinance gspread google-auth gspread_dataframe pandas numpy tqdm

import pandas as pd
import numpy as np
import time
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
import yfinance as yf
from datetime import datetime

# Step 2: Google Authentication
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Step 3: Helper Functions

def to_numeric(val):
    """Safely converts a value to a float, returning 0 if it fails."""
    if val is None or not isinstance(val, (int, float, str)):
        return 0
    try:
        return float(val)
    except (ValueError, TypeError):
        return 0

def score_stock(ticker, ticker_info, rs_rating):
    """Calculates SIQ, CAN SLIM, and GARP scores for a single stock."""
    # --- Robust Data Cleaning ---
    info = {k: to_numeric(v) for k, v in ticker_info.items() if isinstance(v, (int, float, str, type(None)))}

    # --- SIQ (Quality) Score Calculation ---
    revenue_growth = info.get('revenueGrowth', 0)
    gross_margin = info.get('grossMargins', 0)
    total_revenue = info.get('totalRevenue', 1)
    free_cashflow = info.get('freeCashflow', 0)
    enterprise_value = info.get('enterpriseValue', 1)
    total_cash = info.get('totalCash', 0)
    total_debt = info.get('totalDebt', 0)

    fcf_margin = free_cashflow / total_revenue if total_revenue else 0
    rule_of_40 = (revenue_growth + fcf_margin) * 100
    net_cash_ev = (total_cash - total_debt) / enterprise_value if enterprise_value else 0

    # Fetch financial statements for robust ROIC calculation
    roic = 0
    try:
        income_stmt = ticker_info['income_statement']
        balance_sheet = ticker_info['balance_sheet']
        ebit = to_numeric(income_stmt.loc['EBIT'].iloc[0]) if 'EBIT' in income_stmt.index else 0
        debt = to_numeric(balance_sheet.loc['Total Debt'].iloc[0]) if 'Total Debt' in balance_sheet.index else 0
        equity = to_numeric(balance_sheet.loc['Total Stockholder Equity'].iloc[0]) if 'Total Stockholder Equity' in balance_sheet.index else 0
        invested_capital = debt + equity
        roic = ebit / invested_capital if invested_capital > 0 else 0
    except Exception:
        pass # roic remains 0

    siq_score = (
        (1 if revenue_growth > 0.15 else 0) +
        (1 if gross_margin > 0.50 else 0) +
        (1 if rule_of_40 > 40 else 0) +
        (1 if roic > 0.10 else 0) +
        (1 if net_cash_ev > 0 else 0)
    )

    # --- CAN SLIM (Momentum) Score Calculation ---
    q_eps_growth = 0
    annual_eps_growth = 0
    try:
        quarterly_net_income = ticker_info['quarterly_income_statement'].loc['Net Income'].apply(to_numeric)
        q_eps_growth = quarterly_net_income.pct_change(periods=4, fill_method=None).iloc[-1] if len(quarterly_net_income) > 4 else 0

        annual_net_income = ticker_info['income_statement'].loc['Net Income'].apply(to_numeric)
        annual_eps_growth = annual_net_income.pct_change(fill_method=None).iloc[-1] if len(annual_net_income) > 1 else 0
    except Exception:
        pass # growth rates remain 0

    price_vs_52w_high = info.get('currentPrice', 0) / info.get('fiftyTwoWeekHigh', 1)

    can_slim_score = (
        (1 if q_eps_growth > 0.25 else 0) +
        (1 if annual_eps_growth > 0.25 else 0) +
        (1 if price_vs_52w_high > 0.85 else 0) +
        (1 if rs_rating > 80 else 0)
    )

    # --- GARP (Growth at a Reasonable Price) Score Calculation ---
    pe_ratio = info.get('trailingPE', 0)
    forward_eps = info.get('forwardEps', 0)
    trailing_eps = info.get('trailingEps', 1)

    expected_eps_growth = (forward_eps / trailing_eps) - 1 if trailing_eps > 0 and forward_eps > 0 else annual_eps_growth

    peg_ratio = pe_ratio / (expected_eps_growth * 100) if expected_eps_growth > 0 else 999

    garp_score = (
        (2 if annual_eps_growth > 0.20 else (1 if annual_eps_growth > 0.10 else 0)) +
        (1 if revenue_growth > 0.15 else 0) +
        (2 if peg_ratio < 1.0 else (1 if peg_ratio < 1.5 else 0))
    )

    # --- Combine Scores and Data ---
    return {
        'SIQ Score': siq_score,
        'CAN SLIM Score': can_slim_score,
        'GARP Score': garp_score,
        'Combined Score': siq_score + can_slim_score + garp_score,
        'Revenue Growth': revenue_growth,
        'Quarterly EPS Growth': q_eps_growth,
        'Annual EPS Growth': annual_eps_growth,
        'Gross Margin': gross_margin,
        'ROIC': roic,
        'Rule of 40': rule_of_40,
        'RS Rating': rs_rating,
        'PEG Ratio': peg_ratio,
        'Net Cash / EV': net_cash_ev,
        'Price vs 52w High': price_vs_52w_high
    }

def get_all_ticker_info(tickers):
    """Fetches info and financials for all tickers in a more robust way."""
    all_info = {}
    for ticker in tqdm(tickers, desc="Fetching Financial Data"):
        try:
            t = yf.Ticker(ticker)
            info = t.info
            info['income_statement'] = t.income_stmt
            info['quarterly_income_statement'] = t.quarterly_income_stmt
            info['balance_sheet'] = t.balance_sheet
            all_info[ticker] = info
            time.sleep(0.1) # Small pause
        except Exception:
            all_info[ticker] = {}
            continue
    return all_info

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 2 (Triple Scorer) ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Successfully connected to Google Sheet: '{sh.title}'")

    print(f"Reading tickers from the '{INPUT_TAB_NAME}' tab...")
    ws_in = sh.worksheet(INPUT_TAB_NAME)
    tickers_to_process = ws_in.col_values(1)[1:]
    tickers_to_process = [t for t in tickers_to_process if t] # Remove empty strings
    print(f"Found {len(tickers_to_process)} tickers to analyze.")

    print("Fetching historical data for all tickers (for RS Rating)...")
    hist_data = yf.download(tickers_to_process, period="1y", progress=True, auto_adjust=True)['Close']

    print("\nCalculating Relative Strength (RS) Ratings...")
    rs_ratings = (hist_data.iloc[-1] / hist_data.iloc[0] - 1).rank(pct=True) * 100

    print("Fetching detailed financial data for all tickers...")
    all_ticker_info = get_all_ticker_info(tickers_to_process)

    results = []
    for ticker in tqdm(tickers_to_process, desc="Scoring Stocks"):
        if ticker in all_ticker_info and all_ticker_info[ticker]:
            rs_rating = rs_ratings.get(ticker, 0)
            scores = score_stock(ticker, all_ticker_info[ticker], rs_rating)

            base_info = {
                'Ticker': ticker,
                'Company Name': all_ticker_info[ticker].get('shortName', 'N/A'),
                'Market Cap': all_ticker_info[ticker].get('marketCap', 0)
            }
            base_info.update(scores)
            results.append(base_info)

    results_df = pd.DataFrame(results).sort_values(by='Combined Score', ascending=False)

    # Clean the DataFrame of non-JSON compliant values before writing
    results_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    results_df = results_df.astype(object).where(pd.notnull(results_df), None)

    if not results_df.empty:
        print(f"\nWriting {len(results_df)} scores to the '{OUTPUT_TAB_NAME}' tab...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=len(results_df) + 5, cols=len(results_df.columns) + 2)

        set_with_dataframe(ws_out, results_df, include_index=False, include_column_header=True)
        print("✅ Done. Your Triple Score analysis is complete.")
    else:
        print("No stocks could be scored. The output sheet was not updated.")

except gspread.exceptions.WorksheetNotFound as e:
    print(f"❌ ERROR: Input tab '{INPUT_TAB_NAME}' not found. Please ensure Module 1 has run successfully.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 2 Complete ---")

--- Starting Alpha Engine: Module 2 (Triple Scorer) ---
✅ Successfully connected to Google Sheet: 'Stock Tracker'
Reading tickers from the 'Filtered_Universe' tab...


[                       0%                       ]

Found 6 tickers to analyze.
Fetching historical data for all tickers (for RS Rating)...


[*********************100%***********************]  6 of 6 completed


Calculating Relative Strength (RS) Ratings...
Fetching detailed financial data for all tickers...


Fetching Financial Data:   0%|          | 0/6 [00:00<?, ?it/s]

Scoring Stocks:   0%|          | 0/6 [00:00<?, ?it/s]


Writing 5 scores to the 'Triple_Scores' tab...
✅ Done. Your Triple Score analysis is complete.

--- Module 2 Complete ---


In [ ]:
#@title 🚀 Alpha Engine - Module 2.5: The Moat & Management Check (Direct SEC, Throttled & Compliant)
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  Provide a real name and email for the `USER_AGENT` variable.
#@markdown 3.  Add your Gemini API Key to Colab's Secrets Manager with the name `GEMINI_API_KEY`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
USER_AGENT = "Manmohan Sharma redditcommunitydiscussions@gmail.com" #@param {type:"string"}
INPUT_TAB_NAME = "Triple_Scores" #@param {type:"string"}
OUTPUT_TAB_NAME = "Moat_And_Management" #@param {type:"string"}
TOP_N_STOCKS_TO_ANALYZE = 10 #@param {type:"integer"}

# ==============================================================================
# The code below this line executes the analysis logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install yfinance gspread google-auth gspread_dataframe requests pandas numpy tqdm beautifulsoup4 google-generativeai

import pandas as pd
import numpy as np
import time
import json
import re
import requests
import random
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, userdata
from google.auth import default
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import google.generativeai as genai

# Step 2: Google Authentication (for Sheets and Gemini)
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Configure Gemini API
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
except userdata.SecretNotFoundError:
    raise ValueError("Gemini API Key not found. Please add it to Colab's Secrets Manager with the name 'GEMINI_API_KEY'.")

# Step 3: SEC Session (Compliant Headers + Throttling + Retries)
if USER_AGENT == "Your Full Name your.email@example.com":
    raise ValueError("Please provide a real name and email in the USER_AGENT field for SEC compliance.")

BASE_HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Accept": "application/json,text/html;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
}

session = requests.Session()
retry = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET", "HEAD"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update(BASE_HEADERS)

_MIN_INTERVAL = 0.2
_last_call = 0.0

def _polite_get(url, **kwargs):
    """Makes a GET request while respecting rate limits."""
    global _last_call
    now = time.monotonic()
    wait = _last_call + _MIN_INTERVAL - now
    if wait > 0:
        time.sleep(wait)

    r = session.get(url, timeout=30, **kwargs)
    _last_call = time.monotonic()

    if r.status_code in (403, 429):
        retry_after = r.headers.get("Retry-After")
        if retry_after:
            time.sleep(float(retry_after) + 0.1)
        else:
            time.sleep(2 + random.random())
        r = session.get(url, timeout=30, **kwargs)

    r.raise_for_status()
    return r

# Step 4: Helper Functions
_company_tickers_cache = None

def _load_company_tickers():
    global _company_tickers_cache
    if _company_tickers_cache is None:
        print("Downloading master company ticker list from SEC...")
        resp = _polite_get("https://www.sec.gov/files/company_tickers.json")
        _company_tickers_cache = resp.json()
    return _company_tickers_cache

def _cik_for_ticker(ticker):
    t = (ticker or "").strip().upper()
    data = _load_company_tickers()
    for rec in data.values():
        if rec.get("ticker", "").upper() == t:
            return f'{int(rec["cik_str"]):010d}'
    raise ValueError(f"CIK not found for ticker: {ticker}")

def _latest_annual_doc_url(cik10):
    subs = _polite_get(f"https://data.sec.gov/submissions/CIK{cik10}.json").json()
    recent = subs["filings"]["recent"]
    for i, form in enumerate(recent["form"]):
        if form in ("10-K", "20-F", "40-F"):
            accn_nodash = recent["accessionNumber"][i].replace("-", "")
            primary = recent["primaryDocument"][i]
            return f"https://www.sec.gov/Archives/edgar/data/{int(cik10)}/{accn_nodash}/{primary}"
    raise RuntimeError("No recent 10-K/20-F/40-F in 'recent' list")

def _download_filing_text(url):
    r = _polite_get(url)
    soup = BeautifulSoup(r.content, "html.parser")
    return soup.get_text(separator="\n", strip=True)

def _extract_business_and_risks(text, max_len=15000):
    lower = text.lower()
    i1 = lower.find("item 1. business")
    i1a = lower.find("item 1a. risk factors")
    i1b = lower.find("item 1b.")

    if i1 != -1 and i1a != -1:
        business = text[i1:i1a]
        risks = text[i1a:i1b] if i1b != -1 else text[i1a:]
        return (business + "\n" + risks)[:max_len]
    else:
        return text[:max_len]

def get_latest_annual_filing_text_from_sec(ticker):
    try:
        cik10 = _cik_for_ticker(ticker)
        url = _latest_annual_doc_url(cik10)
        raw_text = _download_filing_text(url)
        return _extract_business_and_risks(raw_text)
    except Exception as e:
        return f"Error processing {ticker}: {e}"

def analyze_with_ai(company_name, text_10k):
    """Uses the Gemini API to analyze the 10-K text."""
    model = genai.GenerativeModel('gemini-1.5-flash')
    # --- UPDATED PROMPT ---
    # This new prompt asks for a single, easy-to-read paragraph.
    prompt = f"""
    Act as a world-class investment analyst. I have provided text from the annual report for the company '{company_name}'.

    Your task is to read the text and write a very short, easy-to-read summary (2-4 sentences) for an investor.

    The summary should touch on the company's business model, its main competitive advantage (moat), and the biggest risk it faces.

    Here is the text to analyze:
    ---
    {text_10k}
    ---
    """
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini API call failed: {e}"

def run_analysis(top_stocks):
    rows = []
    for stock in tqdm(top_stocks, desc="Analyzing Annual Reports"):
        try:
            filing_text = get_latest_annual_filing_text_from_sec(stock['Ticker'])
            if "Error processing" in filing_text or "Could not find" in filing_text or "No recent" in filing_text:
                ai_summary = filing_text
            else:
                ai_summary = analyze_with_ai(stock['Company Name'], filing_text)

            rows.append({
                'Ticker': stock['Ticker'], 'Company Name': stock['Company Name'],
                'Sector': stock.get('Sector'), 'Combined Score': stock['Combined Score'],
                'AI Analysis': ai_summary
            })
            # Add a pause after each stock is fully processed to respect Gemini API rate limits.
            time.sleep(2)
        except Exception as e:
            rows.append({
                'Ticker': stock.get('Ticker'), 'Company Name': stock.get('Company Name'),
                'Sector': stock.get('Sector'), 'Combined Score': stock.get('Combined Score'),
                'AI Analysis': f"An unexpected error occurred: {e}"
            })
    return pd.DataFrame(rows)

# Step 5: Main Execution Block
print("--- Starting Alpha Engine: Module 2.5 ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Connected to Google Sheet: '{sh.title}'")

    print(f"Reading top {TOP_N_STOCKS_TO_ANALYZE} stocks from '{INPUT_TAB_NAME}'...")
    ws_in = sh.worksheet(INPUT_TAB_NAME)
    df_scores = pd.DataFrame(ws_in.get_all_records())
    top_stocks_df = df_scores.head(TOP_N_STOCKS_TO_ANALYZE)
    tickers_to_analyze = top_stocks_df.to_dict('records')
    print(f"Found {len(tickers_to_analyze)} stocks to analyze.")

    analysis_df = run_analysis(tickers_to_analyze)

    if not analysis_df.empty:
        print(f"\nWriting {len(analysis_df)} analyses to '{OUTPUT_TAB_NAME}'...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=len(analysis_df) + 5, cols=len(analysis_df.columns) + 2)

        set_with_dataframe(ws_out, analysis_df, include_index=False, include_column_header=True)
        print("✅ Done.")
    else:
        print("No stocks could be analyzed. The output sheet was not updated.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Spreadsheet not found. Please check your SHEET_ID.")
except gspread.exceptions.WorksheetNotFound:
    print(f"❌ ERROR: Input tab '{INPUT_TAB_NAME}' not found in your sheet.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 2.5 Complete ---")


ValueError: Gemini API Key not found. Please add it to Colab's Secrets Manager with the name 'GEMINI_API_KEY'.

In [ ]:
#@title 🚀 Alpha Engine - Module 3: The Catalyst Tracker
#@markdown ---
#@markdown ### 1. Configuration
#@markdown Enter your Google Sheet ID and the required tab names. This module will read your top-ranked stocks from the `Input Tab` and save the final watchlist to the `Output Tab`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
INPUT_TAB_NAME = "Dual_Scores" #@param {type:"string"}
OUTPUT_TAB_NAME = "Catalyst_Watchlist" #@param {type:"string"}
NEWS_API_KEY = "0793746a3a7a416bbffad5c682a9e8cb" #@param {type:"string"}

#@markdown ---
#@markdown ### 2. Catalyst Search Parameters
#@markdown Configure how the catalyst search will run.
TOP_N_STOCKS_TO_SCAN = 50 #@param {type:"integer"}
#@markdown Enter comma-separated keywords to search for in news headlines.
CATALYST_KEYWORDS = "FDA decision, FDA rejection, Phase 3 trial, Phase 2 trial, topline data, clinical trial results, investor day, product launch, earnings guidance, new contract, strategic partnership, regulatory approval, acquisition, merger, patent approval, lawsuit, product recall, earnings beat" #@param {type:"string"}

#@markdown ---
#@markdown ### 3. Click the "Play" button to run the tracker.
#@markdown This will scan the top stocks from your `Dual_Scores` list for catalyst-related news and create a final, focused watchlist.

# ==============================================================================
# The code below this line executes the catalyst tracking logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install and Import Libraries
!pip -q install yfinance gspread google-auth gspread_dataframe requests pandas numpy tqdm newsapi-python

import pandas as pd
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
import time
import numpy as np
from newsapi import NewsApiClient
from datetime import datetime, timedelta

# Step 2: Authenticate with Google
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Step 3: Helper Functions
def find_catalysts(tickers, api_key, keywords):
    """Searches for catalyst-related news for a list of tickers."""
    if not api_key or api_key == "PASTE_YOUR_FREE_NEWSAPI.ORG_KEY_HERE":
        print("⚠️ WARNING: NewsAPI key is missing. Catalyst search will be skipped.")
        return pd.DataFrame()

    newsapi = NewsApiClient(api_key=api_key)
    catalyst_hits = []
    keyword_list = [k.strip().lower() for k in keywords.split(',')]

   # --- FIX ---
    # Search for news in the last 29 days to comply with NewsAPI's free plan limit of 30 days.
    from_date = (datetime.now() - timedelta(days=29)).strftime('%Y-%m-%d')


    for ticker in tqdm(tickers, desc="Scanning for Catalysts"):
        try:
            # The API is better at finding company names than tickers
            # We'll assume the input dataframe has the company name
            query = f'"{ticker["Company Name"]}" OR {ticker["Ticker"]}'

            all_articles = newsapi.get_everything(q=query,
                                                  language='en',
                                                  from_param=from_date,
                                                  sort_by='publishedAt',
                                                  page_size=20)

            for article in all_articles['articles']:
                title = article.get('title', '').lower()
                description = article.get('description', '').lower()

                # Check if any of our keywords are in the headline or description
                if any(keyword in f"{title} {description}" for keyword in keyword_list):
                    catalyst_hits.append({
                        'Ticker': ticker['Ticker'],
                        'Company Name': ticker['Company Name'],
                        'Combined Score': ticker['Combined Score'],
                        'Catalyst Headline': article.get('title'),
                        'Published Date': article.get('publishedAt', '').split('T')[0],
                        'Source': article.get('source', {}).get('name'),
                        'URL': article.get('url')
                    })
                    # Stop after finding the first (most recent) catalyst for this ticker
                    break
            time.sleep(0.5) # Be respectful to the free API
        except Exception as e:
            # Can happen if API rate limit is hit
            print(f"Could not process news for {ticker['Ticker']}: {e}")
            continue

    return pd.DataFrame(catalyst_hits)


# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 3 ---")

try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Successfully connected to Google Sheet: '{sh.title}'")

    # Read the ranked list from the Dual_Scores tab
    print(f"Reading top {TOP_N_STOCKS_TO_SCAN} stocks from the '{INPUT_TAB_NAME}' tab...")
    ws_in = sh.worksheet(INPUT_TAB_NAME)

    # Read into a pandas DataFrame to easily select top N
    df_scores = pd.DataFrame(ws_in.get_all_records())
    top_stocks_df = df_scores.head(TOP_N_STOCKS_TO_SCAN)

    # Convert to list of dictionaries for the catalyst function
    tickers_to_scan = top_stocks_df.to_dict('records')

    print(f"Found {len(tickers_to_scan)} stocks to scan for catalysts.")

    # Find catalysts for the top stocks
    catalyst_df = find_catalysts(tickers_to_scan, NEWS_API_KEY, CATALYST_KEYWORDS)

    if not catalyst_df.empty:
        # Sort by date to see the most recent catalysts first
        catalyst_df = catalyst_df.sort_values(by='Published Date', ascending=False).reset_index(drop=True)

        print(f"\nFound {len(catalyst_df)} stocks with potential near-term catalysts.")

        # Save results to the output tab
        print(f"Writing watchlist to the '{OUTPUT_TAB_NAME}' tab...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(OUTPUT_TAB_NAME, rows=len(catalyst_df)+1, cols=len(catalyst_df.columns)+1)

        set_with_dataframe(ws_out, catalyst_df, include_index=False, include_column_header=True)
        print(f"\n✅ Successfully wrote Catalyst Watchlist to your Google Sheet.")
    else:
        print("\nNo stocks with recent catalyst-related news were found. The output sheet was not updated.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Input tab '{INPUT_TAB_NAME}' not found. Please check the name.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

print("\n--- Module 3 Complete ---")


In [ ]:
#@title 🚀 Alpha Engine - Module 4: The Scuttlebutt Engine
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  Ensure your Gemini API Key is in Colab Secrets as `GEMINI_API_KEY`.
#@markdown 3.  Add your free SerpApi API Key to Colab Secrets as `SERPAPI_KEY`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
INPUT_TAB_NAME = "Moat_And_Management" #@param {type:"string"}
OUTPUT_TAB_NAME = "Scuttlebutt_Analysis" #@param {type:"string"}

#@markdown ---
#@markdown ### 2. Analysis Parameters
#@markdown Configure how many of the top stocks to scan for online sentiment.
TOP_N_STOCKS_TO_SCAN = 20 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Click the "Play" button to run the analysis.
#@markdown This will perform web searches for your top stocks, use Gemini to summarize the findings, and save the results.

# ==============================================================================
# The code below this line executes the analysis logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install gspread google-auth gspread_dataframe pandas numpy tqdm google-generativeai google-search-results

import pandas as pd
import numpy as np
import time
import json
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, userdata
from google.auth import default
import google.generativeai as genai
from serpapi import GoogleSearch

# Step 2: Google Authentication (for Sheets and Gemini)
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Configure APIs
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    SERPAPI_KEY = userdata.get('SERPAPI_KEY')
except userdata.SecretNotFoundError:
    raise ValueError("API Key not found in Colab Secrets. Please make sure both 'GEMINI_API_KEY' and 'SERPAPI_KEY' are set.")

# Step 3: Helper Functions
def perform_scuttlebutt_search(company_name, ticker):
    """Performs targeted Google searches to find relevant discussions."""
    search_queries = [
        f'"{company_name}" reviews reddit',
        f'"{ticker}" stock discussion forum',
        f'"{company_name}" product reviews',
        f'"{company_name}" vs competitor'
    ]

    all_snippets = []
    print(f"INFO: Searching for '{company_name}'...")

    for query in search_queries:
        try:
            params = {
              "q": query,
              "api_key": SERPAPI_KEY,
              "num": 3 # Get top 3 results for each query
            }
            search = GoogleSearch(params)
            results = search.get_dict()

            if 'organic_results' in results:
                for result in results['organic_results']:
                    all_snippets.append(result.get('snippet', ''))
            time.sleep(1) # Be respectful to the API
        except Exception as e:
            print(f"Warning: Search for query '{query}' failed. {e}")
            continue

    return " ".join(all_snippets)

def analyze_scuttlebutt_with_ai(company_name, scuttlebutt_text):
    """Uses Gemini to summarize the scuttlebutt findings."""
    model = genai.GenerativeModel('gemini-1.5-flash')
    prompt = f"""
    Act as a world-class investment research analyst performing "scuttlebutt" research on the company '{company_name}'. I have provided a collection of recent text snippets from Reddit, forums, and review sites.

    Your task is to analyze this text and provide a concise, bullet-point summary of the key themes.

    * **Positive Themes:** What do people consistently praise? (e.g., product quality, customer service, innovation)
    * **Negative Themes:** What are the most common complaints or red flags? (e.g., bugs, high prices, poor management, competition)
    * **Overall Vibe:** Based on the language, is the general sentiment more like a loyal fan base, a frustrated user group, or is it mixed?

    If the text is sparse or uninformative, simply state that.

    Here is the text to analyze:
    ---
    {scuttlebutt_text}
    ---
    """
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini API call failed: {e}"

def run_scuttlebutt_analysis(top_stocks):
    rows = []
    for stock in tqdm(top_stocks, desc="Performing Scuttlebutt Research"):
        try:
            scuttlebutt_text = perform_scuttlebutt_search(stock['Company Name'], stock['Ticker'])

            if not scuttlebutt_text.strip():
                ai_summary = "No relevant online discussions found."
            else:
                ai_summary = analyze_scuttlebutt_with_ai(stock['Company Name'], scuttlebutt_text)

            rows.append({
                'Ticker': stock['Ticker'],
                'Company Name': stock['Company Name'],
                'Sector': stock.get('Sector'),
                'Combined Score': stock['Combined Score'],
                'Scuttlebutt Summary': ai_summary
            })
            time.sleep(2) # Pause to respect Gemini API rate limits
        except Exception as e:
            rows.append({
                'Ticker': stock.get('Ticker'),
                'Company Name': stock.get('Company Name'),
                'Sector': stock.get('Sector'),
                'Combined Score': stock.get('Combined Score'),
                'Scuttlebutt Summary': f"An unexpected error occurred: {e}"
            })
    return pd.DataFrame(rows)

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 4 ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Connected to Google Sheet: '{sh.title}'")

    print(f"Reading top {TOP_N_STOCKS_TO_SCAN} stocks from '{INPUT_TAB_NAME}'...")
    ws_in = sh.worksheet(INPUT_TAB_NAME)
    df_scores = pd.DataFrame(ws_in.get_all_records())
    top_stocks_df = df_scores.head(TOP_N_STOCKS_TO_SCAN)
    tickers_to_analyze = top_stocks_df.to_dict('records')
    print(f"Found {len(tickers_to_analyze)} stocks to analyze.")

    analysis_df = run_scuttlebutt_analysis(tickers_to_analyze)

    if not analysis_df.empty:
        print(f"\nWriting {len(analysis_df)} scuttlebutt analyses to '{OUTPUT_TAB_NAME}'...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=len(analysis_df) + 5, cols=len(analysis_df.columns) + 2)

        set_with_dataframe(ws_out, analysis_df, include_index=False, include_column_header=True)
        print("✅ Done.")
    else:
        print("No stocks could be analyzed. The output sheet was not updated.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Spreadsheet not found. Please check your SHEET_ID.")
except gspread.exceptions.WorksheetNotFound:
    print(f"❌ ERROR: Input tab '{INPUT_TAB_NAME}' not found in your sheet.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 4 Complete ---")


In [ ]:
#@title 🚀 Alpha Engine - Module 5: The Final Briefing
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  Ensure your Gemini API Key is in Colab Secrets as `GEMINI_API_KEY`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
#@markdown ---
#@markdown ### 2. Analysis Parameters
#@markdown Configure how many of the top stocks to create a final brief for.
TOP_N_STOCKS_TO_BRIEF = 10 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Click the "Play" button to run the analysis.
#@markdown This will read all the data from the previous modules and generate a final, synthesized investment brief for your top-ranked stocks.

# ==============================================================================
# The code below this line executes the analysis logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install gspread google-auth gspread_dataframe pandas numpy tqdm google-generativeai

import pandas as pd
import numpy as np
import time
import json
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, userdata
from google.auth import default
import google.generativeai as genai

# Step 2: Google Authentication (for Sheets and Gemini)
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Configure Gemini API
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
except userdata.SecretNotFoundError:
    raise ValueError("Gemini API Key not found. Please add it to Colab's Secrets Manager with the name 'GEMINI_API_KEY'.")

# Step 3: Helper Functions
def synthesize_brief_with_ai(stock_data):
    """Uses Gemini to create a final investment brief from all collected data."""
    model = genai.GenerativeModel('gemini-1.5-flash')

    # Consolidate all the data into a structured text block for the prompt
    prompt_data = f"""
    Company: {stock_data.get('Company Name')} ({stock_data.get('Ticker')})
    Sector: {stock_data.get('Sector')}

    Quantitative Scores:
    - Combined Score: {stock_data.get('Combined Score')}
    - SIQ (Quality) Score: {stock_data.get('SIQ Score')}
    - CAN SLIM (Momentum) Score: {stock_data.get('CAN SLIM Score')}

    Moat & Management Analysis (from 10-K):
    {stock_data.get('AI Analysis', 'N/A')}

    Recent Catalysts:
    {stock_data.get('Catalyst Headlines', 'No recent catalyst news found.')}

    Scuttlebutt Summary (from forums/Reddit):
    {stock_data.get('Scuttlebutt Summary', 'No scuttlebutt data found.')}
    """

    prompt = f"""
    Act as a senior investment analyst at a top-tier hedge fund. I have provided you with a structured data packet containing quantitative scores, a qualitative analysis of the company's annual report, recent news catalysts, and a summary of online discussions for a single stock.

    Your task is to synthesize all of this information into a final, comprehensive investment brief. The brief should be structured into three clear sections:

    1.  **Investment Thesis (The Bull Case):** In 3-4 bullet points, explain the primary reasons to be bullish on this stock. Connect the quantitative scores (e.g., high growth) with the qualitative findings (e.g., a strong moat).
    2.  **Key Risks (The Bear Case):** In 3-4 bullet points, outline the most significant risks. Use the 10-K risk factors and any negative themes from the scuttlebutt analysis.
    3.  **Final Verdict & Actionable Plan:** In a short paragraph, provide a final verdict. Is this a high-conviction idea? A speculative bet? A stock to watch? Suggest a potential action (e.g., "Initiate a starter position," "Add to watchlist," "Avoid due to risks").

    Here is the data packet to analyze:
    ---
    {prompt_data}
    ---
    """
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Gemini API call failed: {e}"

def run_final_briefing(top_stocks_data):
    rows = []
    for stock in tqdm(top_stocks_data, desc="Generating Final Briefs"):
        try:
            final_brief = synthesize_brief_with_ai(stock)
            rows.append({
                'Ticker': stock['Ticker'],
                'Company Name': stock['Company Name'],
                'Final Briefing': final_brief
            })
            time.sleep(2) # Pause to respect Gemini API rate limits
        except Exception as e:
            rows.append({
                'Ticker': stock.get('Ticker'),
                'Company Name': stock.get('Company Name'),
                'Final Briefing': f"An unexpected error occurred: {e}"
            })
    return pd.DataFrame(rows)

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 5 ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Connected to Google Sheet: '{sh.title}'")

    # Read data from all previous module tabs
    print("Reading data from previous modules...")
    ws_scores = sh.worksheet("Dual_Scores")
    ws_moat = sh.worksheet("Moat_And_Management")
    ws_catalyst = sh.worksheet("Catalyst_Watchlist")
    ws_scuttlebutt = sh.worksheet("Scuttlebutt_Analysis")

    df_scores = pd.DataFrame(ws_scores.get_all_records())
    df_moat = pd.DataFrame(ws_moat.get_all_records())
    df_catalyst = pd.DataFrame(ws_catalyst.get_all_records())
    df_scuttlebutt = pd.DataFrame(ws_scuttlebutt.get_all_records())

    # Merge all dataframes together based on the Ticker
    # Start with the top N stocks from the main scores sheet
    top_stocks_df = df_scores.head(TOP_N_STOCKS_TO_BRIEF)

    # Merge with other data, keeping all the top stocks
    merged_df = pd.merge(top_stocks_df, df_moat, on='Ticker', how='left', suffixes=('', '_moat'))
    merged_df = pd.merge(merged_df, df_catalyst, on='Ticker', how='left', suffixes=('', '_catalyst'))
    merged_df = pd.merge(merged_df, df_scuttlebutt, on='Ticker', how='left', suffixes=('', '_scuttlebutt'))

    # Clean up duplicate columns from merges
    merged_df = merged_df.loc[:,~merged_df.columns.duplicated()]

    tickers_to_brief = merged_df.to_dict('records')
    print(f"Found {len(tickers_to_brief)} top stocks to create a final brief for.")

    analysis_df = run_final_briefing(tickers_to_brief)

    if not analysis_df.empty:
        output_tab_name = "Final_Briefing"
        print(f"\nWriting {len(analysis_df)} final briefs to '{output_tab_name}'...")
        try:
            ws_out = sh.worksheet(output_tab_name)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=output_tab_name, rows=len(analysis_df)*10, cols=5) # Allocate more rows for long text

        # --- FIX for 400 APIError ---
        # gspread requires a list of lists for all update operations, even for a single cell.
        ws_out.update('A1', [['Ticker']])
        ws_out.update('B1', [['Company Name']])
        ws_out.update('C1', [['Final Briefing']])
        for i, row in analysis_df.iterrows():
            ws_out.update(f'A{i+2}', [[row['Ticker']]])
            ws_out.update(f'B{i+2}', [[row['Company Name']]])
            ws_out.update(f'C{i+2}', [[row['Final Briefing']]])

        print("✅ Done.")
    else:
        print("No stocks could be analyzed. The output sheet was not updated.")

except gspread.exceptions.SpreadsheetNotFound as e:
    print(f"❌ ERROR: A required input tab was not found. Please ensure 'Dual_Scores', 'Moat_And_Management', 'Catalyst_Watchlist', and 'Scuttlebutt_Analysis' tabs exist. Missing tab: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 5 Complete ---")


In [ ]:
#@title 🚀 Alpha Engine - Module 6: The Daily Idea Tracker
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  This module will now automatically read tickers from the output of the previous module (`Final_Briefing`).
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
INPUT_TAB_NAME = "Final_Briefing" #@param {type:"string"}
SCORES_TAB_NAME = "Dual_Scores" #@param {type:"string"}
OUTPUT_TAB_NAME = "Daily_Tracker" #@param {type:"string"}
BENCHMARK_TICKER = "SPY" #@param {type:"string"}

#@markdown ---
#@markdown ### 2. Click the "Play" button to run the tracker.
#@markdown This will track the performance of the ideas generated by your Alpha Engine and forecast a target 10x date.

# ==============================================================================
# The code below this line executes the tracking logic.
# You do not need to edit it.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install yfinance gspread google-auth gspread_dataframe pandas numpy tqdm

import pandas as pd
import numpy as np
import time
import math
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
import yfinance as yf
from datetime import datetime, timedelta

# Step 2: Google Authentication
auth.authenticate_user()
SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds, _ = default(scopes=SCOPES)
gc = gspread.authorize(creds)

# Step 3: Helper Functions
def track_ideas(ideas_df, scores_df, benchmark):
    """Fetches latest data and calculates performance metrics for the generated ideas."""

    tickers = ideas_df['Ticker'].tolist()
    all_tickers = tickers + [benchmark]

    print(f"Fetching latest data for {len(tickers)} stocks and benchmark {benchmark}...")
    hist_data = yf.download(all_tickers, period="2mo", progress=False, auto_adjust=True)['Close']

    tracking_results = []

    for index, row in tqdm(ideas_df.iterrows(), total=ideas_df.shape[0], desc="Tracking Ideas"):
        ticker = row['Ticker']
        company_name = row['Company Name']

        try:
            # Get current data
            current_price = hist_data[ticker].iloc[-1]

            # Technical Health Check: Price vs. 50-day Moving Average
            ma50 = hist_data[ticker].rolling(window=50).mean().iloc[-1]
            price_vs_ma50 = "Above" if current_price > ma50 else "Below"

            # Performance vs. Benchmark (last 30 days)
            if len(hist_data[ticker].dropna()) > 22 and len(hist_data[benchmark].dropna()) > 22:
                stock_perf_30d = (hist_data[ticker].iloc[-1] / hist_data[ticker].iloc[-22]) - 1
                bench_perf_30d = (hist_data[benchmark].iloc[-1] / hist_data[benchmark].iloc[-22]) - 1
                outperformance_30d = stock_perf_30d - bench_perf_30d
            else:
                outperformance_30d = np.nan

            # --- New: Calculate Target 10x Date ---
            target_10x_date = "N/A"
            ticker_scores = scores_df[scores_df['Ticker'] == ticker]

            if not ticker_scores.empty:
                score_row = ticker_scores.iloc[0]
                # Safely convert to numeric, coercing errors to 0
                g = pd.to_numeric(score_row.get('Revenue Growth'), errors='coerce') or 0
                rule_of_40 = pd.to_numeric(score_row.get('Rule of 40'), errors='coerce') or 0

                # Derive FCF Margin from Rule of 40
                fcfm = (rule_of_40 / 100) - g

                # Quality-adjusted growth rate
                sg = 0.6 * g + 0.4 * fcfm
                sg = max(0.05, min(0.50, sg)) # Clamp growth rate to be realistic (5% to 50%)

                if sg > 0:
                    years_to_10x = math.log(10) / math.log(1 + sg)
                    target_date = datetime.now() + timedelta(days=365.25 * years_to_10x)
                    target_10x_date = target_date.strftime('%Y-%m-%d')

            tracking_results.append({
                'Ticker': ticker,
                'Company Name': company_name,
                'Current Price': current_price,
                'Target 10x Date': target_10x_date,
                'Price vs 50d MA': price_vs_ma50,
                '1-Month Outperformance vs SPY': f"{outperformance_30d:.2%}" if not np.isnan(outperformance_30d) else "N/A"
            })

        except Exception as e:
            print(f"Warning: Could not process {ticker}. Error: {e}")
            continue

    return pd.DataFrame(tracking_results)

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 6 ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Connected to Google Sheet: '{sh.title}'")

    # Read the finalized ideas and the scores data
    print(f"Reading your finalized ideas from the '{INPUT_TAB_NAME}' tab...")
    ws_in = sh.worksheet(INPUT_TAB_NAME)
    ideas_df = pd.DataFrame(ws_in.get_all_records())

    print(f"Reading growth data from the '{SCORES_TAB_NAME}' tab...")
    ws_scores = sh.worksheet(SCORES_TAB_NAME)
    scores_df = pd.DataFrame(ws_scores.get_all_records())

    # Ensure required columns exist
    if 'Ticker' not in ideas_df.columns:
        raise ValueError(f"The '{INPUT_TAB_NAME}' tab must contain a 'Ticker' column.")

    print(f"Found {len(ideas_df)} ideas to track.")

    # Track the ideas
    tracking_df = track_ideas(ideas_df, scores_df, BENCHMARK_TICKER)

    if not tracking_df.empty:
        print(f"\nWriting daily tracking data to the '{OUTPUT_TAB_NAME}' tab...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=len(tracking_df) + 5, cols=len(tracking_df.columns) + 2)

        set_with_dataframe(ws_out, tracking_df, include_index=False, include_column_header=True)
        print("✅ Done. Your daily dashboard is updated.")
    else:
        print("No stocks could be tracked. The output sheet was not updated.")

except gspread.exceptions.WorksheetNotFound as e:
    print(f"❌ ERROR: A required input tab was not found. Please ensure '{INPUT_TAB_NAME}' and '{SCORES_TAB_NAME}' tabs exist. Missing tab: {e.args[0]}")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 6 Complete ---")


In [ ]:
#@title 🚀 Alpha Engine - Module 7: The Investor's Briefing
#@markdown ---
#@markdown ### 1. Configuration
#@markdown **Action Required:**
#@markdown 1.  Enter your Google Sheet ID.
#@markdown 2.  Ensure your `GEMINI_API_KEY` is saved in Colab Secrets (the "Key" icon on the left).
#@markdown 3.  This module reads data from all previous module outputs.
#@markdown 4.  The final, comprehensive reports will be saved to a new tab: `Investor_Briefing`.
SHEET_ID = "1fAX4EWbDLUrYbDGOFyhvIiWPymI_91I-vRi8vLxIpZU" #@param {type:"string"}
TOP_N_STOCKS_TO_BRIEF = 5 #@param {type:"integer"}

#@markdown ---
#@markdown ### 2. Click the "Play" button to generate the final briefs.
#@markdown This will synthesize all data into a professional, one-page report for your top stocks.

# ==============================================================================
# The code below this line executes the analysis logic.
# ==============================================================================

# Step 1: Install & Imports
!pip -q install yfinance gspread google-auth gspread_dataframe pandas numpy tqdm google-generativeai

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, userdata
from google.auth import default
import time
import google.generativeai as genai
import json

# Step 2: Google & Gemini Authentication
try:
    auth.authenticate_user()
    SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
    creds, _ = default(scopes=SCOPES)
    gc = gspread.authorize(creds)
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
except userdata.SecretNotFoundError:
    raise ValueError("Gemini API Key not found. Please add it to Colab Secrets (the 'Key' icon on the left) with the name 'GEMINI_API_KEY'.")
except Exception as e:
    raise RuntimeError(f"An error occurred during authentication: {e}")

# Step 3: Helper Functions
def get_data_from_sheet(sh, tab_name):
    """Fetches data from a specific tab and returns it as a DataFrame."""
    try:
        ws = sh.worksheet(tab_name)
        return pd.DataFrame(ws.get_all_records())
    except gspread.exceptions.WorksheetNotFound:
        print(f"Warning: Tab '{tab_name}' not found. Skipping.")
        return pd.DataFrame()

def generate_investor_brief(stock_data):
    """Uses Gemini to synthesize all collected data into the new investor brief format."""
    model = genai.GenerativeModel('gemini-1.5-flash')

    # Prepare quantitative data for the prompt
    quant_summary = f"""
    - Revenue Growth (YoY): {stock_data.get('Revenue Growth', 'N/A'):.1%}
    - Quarterly EPS Growth: {stock_data.get('Quarterly EPS Growth', 'N/A'):.1%}
    - Gross Margin: {stock_data.get('Gross Margin', 'N/A'):.1%}
    - ROIC: {stock_data.get('ROIC', 'N/A'):.1%}
    - PEG Ratio: {stock_data.get('PEG Ratio', 'N/A'):.2f}
    - Price vs 50d MA: {stock_data.get('Price vs 50d MA', 'N/A'):.1%}
    - Relative Strength (RS) Rating: {stock_data.get('RS Rating', 'N/A')}
    """

    prompt = f"""
    Act as a senior investment analyst at a top-tier hedge fund. Your task is to synthesize the provided data into a comprehensive, one-page investment brief for '{stock_data.get('Company Name')} ({stock_data.get('Ticker')})'.
    The output must follow the specified four-part structure precisely.

    **Collected Intelligence:**
    ---
    **1. Quantitative Data:**
    {quant_summary}

    **2. Qualitative Analysis (from 10-K):**
    {stock_data.get('AI Analysis', 'No 10-K analysis available.')}

    **3. Recent News & Catalysts:**
    {stock_data.get('Catalyst Headlines', 'No specific catalysts found in recent news.')}

    **4. Scuttlebutt (Online Sentiment):**
    {stock_data.get('Scuttlebutt Summary', 'No scuttlebutt analysis available.')}
    ---

    **Required Output Format:**
    ---
    **Lens Table**
    Create a markdown table with the columns: "Lens", "What we found", and "Tilt (Positive / Neutral / Negative)".
    - For quantitative lenses (Growth, Profitability, Valuation, Technicals), use the provided data to write a concise summary.
    - For qualitative lenses (Regulatory Risk, Key Opportunities), synthesize insights from the 10-K and news.

    **One-Page Thesis**
    - **Why Bulls Stay Interested:** 2-3 bullet points.
    - **Why Bears Shrug / What Might Go Wrong:** 2-3 bullet points.

    **Probability Sketch (12-Month View)**
    Create a markdown table with the columns: "Scenario", "Approx % Odds", and "Price Target / Drivers". Include Base, Optimistic, and Bear cases.

    **Takeaway & Action Framework**
    - **Bottom Line Tilt:** State the overall tilt (e.g., Positive, Neutral-Positive, Cautious).
    - **Action Idea:** Provide a brief, actionable suggestion for a potential investor.
    ---
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"An error occurred during AI generation: {e}"

# Step 4: Main Execution Block
print("--- Starting Alpha Engine: Module 5 (Investor Briefing) ---")
try:
    sh = gc.open_by_key(SHEET_ID)
    print(f"✅ Connected to Google Sheet: '{sh.title}'")

    print("Reading data from all previous modules...")
    df_scores = get_data_from_sheet(sh, "Triple_Scores")
    df_moat = get_data_from_sheet(sh, "Moat_And_Management")
    df_catalyst = get_data_from_sheet(sh, "Catalyst_Watchlist")
    df_scuttlebutt = get_data_from_sheet(sh, "Scuttlebutt_Analysis")
    # Also get daily tracker data for technicals
    df_tracker = get_data_from_sheet(sh, "Daily_Tracker")


    if df_scores.empty:
        raise ValueError("The 'Triple_Scores' tab is empty. Please run Module 2 first.")

    # Merge all data into a single DataFrame for easy lookup
    merged_df = df_scores.merge(df_moat, on='Ticker', how='left', suffixes=('', '_moat'))
    if not df_catalyst.empty:
        merged_df = merged_df.merge(df_catalyst, on='Ticker', how='left', suffixes=('', '_catalyst'))
    if not df_scuttlebutt.empty:
        merged_df = merged_df.merge(df_scuttlebutt, on='Ticker', how='left', suffixes=('', '_scuttlebutt'))
    if not df_tracker.empty:
         # Ensure we only merge the columns we need to avoid conflicts
        tracker_cols = ['Ticker', 'Price vs 50d MA', '1M Relative to SPY']
        merged_df = merged_df.merge(df_tracker[tracker_cols], on='Ticker', how='left', suffixes=('', '_tracker'))


    top_stocks = merged_df.head(TOP_N_STOCKS_TO_BRIEF).to_dict('records')
    print(f"Found {len(top_stocks)} top stocks to create a final brief for.")

    briefing_results = []
    for stock in tqdm(top_stocks, desc="Generating Investor Briefs"):
        brief = generate_investor_brief(stock)
        briefing_results.append({
            'Ticker': stock['Ticker'],
            'Company Name': stock['Company Name'],
            'Investor Briefing': brief
        })
        time.sleep(2.5) # Pause to respect Gemini API rate limits

    briefing_df = pd.DataFrame(briefing_results)

    if not briefing_df.empty:
        print(f"\nWriting {len(briefing_df)} final briefs to '{OUTPUT_TAB_NAME}'...")
        try:
            ws_out = sh.worksheet(OUTPUT_TAB_NAME)
            ws_out.clear()
        except gspread.exceptions.WorksheetNotFound:
            ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=len(briefing_df) * 20, cols=5)

        # Write headers
        ws_out.update(values=[briefing_df.columns.values.tolist()], range_name='A1')
        # Write data
        ws_out.update(values=briefing_df.values.tolist(), range_name='A2')

        print("✅ Done.")
    else:
        print("No briefs were generated.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Spreadsheet not found. Check SHEET_ID.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

print("\n--- Module 5 Complete ---")

